# Task 1: Multi-Class Benchmark Classification Pipeline (Iris Dataset)

This Jupyter notebook implements an end-to-end supervised multi-class classification pipeline using **Scikit-Learn**, **Pandas**, **Seaborn**, and **Matplotlib**.

--- 
### 📌 Pipeline Objectives:
1. **Environment Setup & Data Ingestion**: Load the Iris benchmark classification dataset and construct a structured Pandas DataFrame.
2. **Exploratory Data Analysis (EDA)**: Visualize pairwise feature distributions and class separability using Seaborn pairplots.
3. **Data Preprocessing & Normalization**: Standardize features using `StandardScaler` (zero mean, unit variance) fitted exclusively on training data to prevent data leakage.
4. **Stratified Data Partitioning**: Split the dataset into 80% training and 20% test subsets with stratification.
5. **Supervised Model Training**: Train Logistic Regression and Decision Tree classifiers.
6. **Performance Benchmarking & Visualization**: Evaluate overall accuracy, precision, recall, F1-scores, plot confusion matrix heatmaps, and visualize decision boundaries.

## 1. Environment Setup & Library Imports
We import core data manipulation packages, visual plotting libraries, and Scikit-Learn modules for preprocessing, training, and evaluation.

In [ ]:
# Scientific computing and DataFrame manipulation
import numpy as np
import pandas as pd

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Scikit-Learn data, preprocessing, models, and metric utilities
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set inline plotting aesthetics
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

## 2. Dataset Loading & Inspection
We ingest the classic Iris dataset, map target numerical classes to target species names, and verify class balance.

In [ ]:
# Load Iris benchmark dataset
iris_data = load_iris()

# Extract features (X) and target labels (y)
X = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
y = pd.Series(iris_data.target, name="target")

# Combine into a single DataFrame for exploratory analysis
df_iris = X.copy()
df_iris['species'] = y.map(dict(enumerate(iris_data.target_names)))

# Print structural metadata
print(f"Dataset Dimension: {X.shape[0]} rows x {X.shape[1]} features")
print(f"Target Species: {list(iris_data.target_names)}")
print("\nTarget Class Distribution:")
print(df_iris['species'].value_counts())

# Preview first 5 rows
df_iris.head()

## 3. Exploratory Data Analysis (EDA)
Pairwise feature distributions reveal distinct linear separability for *Iris-setosa*, while *versicolor* and *virginica* exhibit mild feature overlap.

In [ ]:
# Generate pairwise feature scatter plot matrix with KDE diagonals
pairplot_fig = sns.pairplot(df_iris, hue='species', palette='Set1', diag_kind='kde', markers=["o", "s", "D"])
pairplot_fig.fig.suptitle("Exploratory Pairplot: Iris Feature Distributions", y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Generate correlation heatmap for features
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Train/Test Data Partitioning (80/20 Stratified Split)
We apply stratified sampling to ensure equal class proportions (30 instances per class in training, 10 in test).

In [ ]:
# Partition dataset into 80% Training and 20% Testing subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set samples: {X_train.shape[0]}")
print(f"Testing set samples:  {X_test.shape[0]}")

## 5. Feature Scaling & Normalization
Features are standardized ($\mu=0, \sigma=1$) using `StandardScaler`. Fitting is strictly performed on `X_train` to avoid data leakage.

In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit scaler exclusively on training features, transform training and testing sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify standardization parameters
print("Scaled Training Mean (per feature):\n", np.round(X_train_scaled.mean(axis=0), 4))
print("Scaled Training Std Dev (per feature):\n", np.round(X_train_scaled.std(axis=0), 4))

## 6. Supervised Model Training & Metrics Evaluation
We train **Logistic Regression** and a **Decision Tree Classifier**, evaluating precision, recall, F1-scores, and plotting confusion matrix heatmaps.

In [ ]:
# Instantiate classifiers
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=200),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42, max_depth=4)
}

# Train, predict, and evaluate each classifier
for name, model in models.items():
    # Model Training
    model.fit(X_train_scaled, y_train)
    
    # Inference on Test Dataset
    y_pred = model.predict(X_test_scaled)
    
    # Accuracy evaluation
    acc = accuracy_score(y_test, y_pred)
    
    print("=" * 60)
    print(f"Model Benchmark: {name}")
    print(f"Overall Test Accuracy: {acc * 100:.2f}%\n")
    print("Detailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=iris_data.target_names))
    
    # Plot Confusion Matrix Heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=iris_data.target_names, yticklabels=iris_data.target_names,
                cbar=False, linewidths=1)
    plt.title(f"Confusion Matrix — {name}", fontsize=12, fontweight='bold')
    plt.xlabel("Predicted Label", fontsize=10)
    plt.ylabel("True Label", fontsize=10)
    plt.tight_layout()
    plt.show()

## 7. Decision Boundary Visualization (Petal Features)
To visually verify decision boundaries, we train a 2D Logistic Regression using only **Petal Length** and **Petal Width**.

In [ ]:
# Select 2 key features: Petal Length (col index 2) and Petal Width (col index 3)
X_2d = X.iloc[:, [2, 3]].values
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.20, random_state=42, stratify=y
)

# Scale 2D subset
scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
X_test_2d_scaled = scaler_2d.transform(X_test_2d)

# Fit 2D model
clf_2d = LogisticRegression(random_state=42)
clf_2d.fit(X_train_2d_scaled, y_train_2d)

# Meshgrid generation for boundary plotting
x_min, x_max = X_train_2d_scaled[:, 0].min() - 0.5, X_train_2d_scaled[:, 0].max() + 0.5
y_min, y_max = X_train_2d_scaled[:, 1].min() - 0.5, X_train_2d_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Predict mesh grid values
Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot boundaries
plt.figure(figsize=(9, 6))
cmap_light = ListedColormap(['#FFBBBB', '#BBFFBB', '#BBBBFF'])
cmap_bold = ['red', 'green', 'blue']

plt.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
for idx, cl in enumerate(np.unique(y_train_2d)):
    plt.scatter(x=X_train_2d_scaled[y_train_2d == cl, 0], 
                y=X_train_2d_scaled[y_train_2d == cl, 1],
                c=cmap_bold[idx], label=iris_data.target_names[cl], edgecolors='k', s=40)

plt.xlabel('Standardized Petal Length (cm)', fontsize=11)
plt.ylabel('Standardized Petal Width (cm)', fontsize=11)
plt.title('Logistic Regression Decision Boundaries (2D Petal Features)', fontsize=13, fontweight='bold')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()